In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
#import geopandas
import tensorflow
import keras
from keras.models import Model
from keras.layers import LSTM, Activation, Dense, Dropout, Input, Embedding
from keras.optimizers import RMSprop
#from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing import sequence
#from keras.utils import to_categorical
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
import numpy as np
import math
import sklearn.metrics
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='geewildfires')

T4 GPU works well without costing a lot, L100 faster but expensive


In [ ]:
def open_file(gcs_path):
    #lnglat = None
    #full_file = '/vsigs/jk_wildfiresspreadts/' + gcs_path
    full_file = 'gs://jk_wildfiresspreadts/' + gcs_path
    df = pd.read_csv(full_file)
    return df

In [ ]:
fires_df = open_file("FINAL_PartFourOutput_WUI.csv")
fires_df.head()

,Unnamed: 0,FIRE_ID,UrbanAngle,occur_id,point_id,IsUrban,WUIBreach,Month,NDVI,NDWI,NBR,DEM,aspect,hillshade,slope,bi,erc,eto,fm100,fm1000,pr,rmax,rmin,th,pop_density,tmmn,tmmx,vpd,vs,LandCover,Y,Water,Very_low_rural,Low_density_rural,Rural_cluster,Suburban,Semi_dense_urban,Dense_urban,Urban_centre,first_urban_point,RemoveFlag
0,202,NV4091811770219850817,-52.270315,2,0,0,0,8,0.056068,0.105315,-0.133341,1580.415405,215.0,195.0,8.0,-0.000004,59.422588,6.077752,6.004851,8.000117,0.76846,39.612396,12.523386,37.000031,0.0,285.6987,303.04364,2.164311,2.565506,71,1,0,1,0,0,0,0,0,0,99,False
1,203,NV4091811770219850817,-52.270315,2,1,0,0,8,0.060680,0.103729,-0.121322,1580.415405,215.0,195.0,8.0,-0.000004,59.422588,6.077752,6.004851,8.000117,0.76846,39.612396,12.523386,37.000031,0.0,285.6987,303.04364,2.164311,2.565506,71,1,0,1,0,0,0,0,0,0,99,False
2,204,NV4091811770219850817,-52.270315,2,2,0,0,8,0.060265,0.100935,-0.134430,1578.098389,276.0,189.0,3.0,-0.000004,59.422588,6.077752,6.004851,8.000117,0.76846,39.612396,12.523386,37.000031,0.0,285.6987,303.04364,2.164311,2.565506,71,1,0,1,0,0,0,0,0,0,99,False
3,205,NV4091811770219850817,-52.270315,2,3,0,0,8,0.056502,0.098674,-0.140869,1575.750000,269.0,189.0,3.0,-0.000004,59.422588,6.077752,6.004851,8.000117,0.76846,39.612396,12.523386,37.000031,0.0,285.6987,303.04364,2.164311,2.565506,71,1,0,1,0,0,0,0,0,0,99,False
4,206,NV4091811770219850817,-52.270315,2,4,0,0,8,0.065172,0.102284,-0.157150,1575.750000,269.0,189.0,3.0,-0.000004,59.422588,6.077752,6.004851,8.000117,0.76846,39.612396,12.523386,37.000031,0.0,285.6987,303.04364,2.164311,2.565506,71,1,0,1,0,0,0,0,0,0,99,False


In [ ]:
print(fires_df.info(verbose=True, show_counts=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1369333 entries, 0 to 1369332
Data columns (total 41 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   Unnamed: 0         1369333 non-null  int64  
 1   FIRE_ID            1369333 non-null  object 
 2   UrbanAngle         1369333 non-null  float64
 3   occur_id           1369333 non-null  int64  
 4   point_id           1369333 non-null  int64  
 5   IsUrban            1369333 non-null  int64  
 6   WUIBreach          1369333 non-null  int64  
 7   Month              1369333 non-null  int64  
 8   NDVI               1369333 non-null  float64
 9   NDWI               1369333 non-null  float64
 10  NBR                1369333 non-null  float64
 11  DEM                1369333 non-null  float64
 12  aspect             1369333 non-null  float64
 13  hillshade          1369333 non-null  float64
 14  slope              1369333 non-null  float64
 15  bi                 1369333 non-n

In [ ]:
fires = fires_df.drop(columns=["Unnamed: 0"])
fires.head(50)

,FIRE_ID,UrbanAngle,occur_id,point_id,IsUrban,WUIBreach,Month,NDVI,NDWI,NBR,DEM,aspect,hillshade,slope,bi,erc,eto,fm100,fm1000,pr,rmax,rmin,th,pop_density,tmmn,tmmx,vpd,vs,LandCover,Y,Water,Very_low_rural,Low_density_rural,Rural_cluster,Suburban,Semi_dense_urban,Dense_urban,Urban_centre,first_urban_point,RemoveFlag
0,NV4091811770219850817,-52.270315,2,0,0,0,8,0.056068,0.105315,-0.133341,1580.415405,215.0,195.0,8.0,-0.000004,59.422588,6.077752,6.004851,8.000117,0.768460,39.612396,12.523386,37.000031,0.0,285.698700,303.043640,2.164311,2.565506,71,1,0,1,0,0,0,0,0,0,99,False
1,NV4091811770219850817,-52.270315,2,1,0,0,8,0.060680,0.103729,-0.121322,1580.415405,215.0,195.0,8.0,-0.000004,59.422588,6.077752,6.004851,8.000117,0.768460,39.612396,12.523386,37.000031,0.0,285.698700,303.043640,2.164311,2.565506,71,1,0,1,0,0,0,0,0,0,99,False
2,NV4091811770219850817,-52.270315,2,2,0,0,8,0.060265,0.100935,-0.134430,1578.098389,276.0,189.0,3.0,-0.000004,59.422588,6.077752,6.004851,8.000117,0.768460,39.612396,12.523386,37.000031,0.0,285.698700,303.043640,2.164311,2.565506,71,1,0,1,0,0,0,0,0,0,99,False
3,NV4091811770219850817,-52.270315,2,3,0,0,8,0.056502,0.098674,-0.140869,1575.750000,269.0,189.0,3.0,-0.000004,59.422588,6.077752,6.004851,8.000117,0.768460,39.612396,12.523386,37.000031,0.0,285.698700,303.043640,2.164311,2.565506,71,1,0,1,0,0,0,0,0,0,99,False
4,NV4091811770219850817,-52.270315,2,4,0,0,8,0.065172,0.102284,-0.157150,1575.750000,269.0,189.0,3.0,-0.000004,59.422588,6.077752,6.004851,8.000117,0.768460,39.612396,12.523386,37.000031,0.0,285.698700,303.043640,2.164311,2.565506,71,1,0,1,0,0,0,0,0,0,99,False
5,NV4091811770219850817,-52.270315,2,5,0,0,8,0.061031,0.097288,-0.157150,1569.649536,317.0,190.0,5.0,-0.000004,59.422588,6.077752,6.004851,8.000117,0.768460,39.612396,12.523386,37.000031,0.0,285.698700,303.043640,2.164311,2.565506,71,1,0,1,0,0,0,0,0,0,99,False
6,NV4091811770219850817,-52.270315,2,6,0,0,8,0.065087,0.103098,-0.152231,1569.649536,317.0,190.0,5.0,-0.000004,63.949570,6.403963,5.500517,7.037678,0.522566,38.585674,10.066293,37.000031,0.0,285.359467,305.917786,2.564676,2.497403,71,1,0,1,0,0,0,0,0,0,99,False
7,NV4091811770219850817,-52.270315,2,7,0,0,8,0.054541,0.095758,-0.182467,1566.874756,308.0,191.0,4.0,-0.000004,63.949570,6.403963,5.500517,7.037678,0.522566,38.585674,10.066293,37.000031,0.0,285.359467,305.917786,2.564676,2.497403,71,1,0,1,0,0,0,0,0,0,99,False
8,NV4091811770219850817,-52.270315,2,8,0,0,8,0.053248,0.089074,-0.198369,1544.093262,3.0,178.0,13.0,-0.000004,63.949570,6.403963,5.500517,7.037678,0.522566,38.585674,10.066293,37.000031,0.0,285.359467,305.917786,2.564676,2.497403,71,1,0,1,0,0,0,0,0,0,99,False
9,NV4091811770219850817,-52.270315,2,9,0,0,8,0.063477,0.092551,-0.163862,1544.093262,3.0,178.0,13.0,-0.000004,63.949570,6.403963,5.500517,7.037678,0.522566,38.585674,10.066293,37.000031,0.0,285.359467,305.917786,2.564676,2.497403,71,1,0,1,0,0,0,0,0,0,99,False


In [ ]:
# Initial order:
# 1. count sequence lengths and number of sequences (create_sequences_for_training)
# 2. drop column names (create_sequences_for_training)
# 2. split into X and Y as lists of sequences (create_sequences_for_training)
# 3. split into train and test (using the lists - count lists, take first 66% as train) (split_train_test)
# 4. convert from df to np array (.values)
# 5. concatenate all values in the X training array (train_scaler)
# 6. .fit the StandardScaler on those values (train_scaler)
# 7. .transform all the X values per sequence using the new scaler object (normalise X) (scaler_transform)
# 8. pad sequences (sequence.pad_sequences)
# 9. reshape for keras

# Order changed again later - train / test split done on FIRE_ID, then lists of sequences created

In [ ]:
def create_fire_id_groups(df):
    fire_groups = df.groupby('FIRE_ID')
    fire_list = []
    for fidx, fseq in fire_groups:
        fire_list.append(fseq)
    return fire_list

def create_sequences_from_fire_ids(list_of_fires):
    no_sequences = 0
    max_seq = 0
    min_seq = 100
    longest_id = 0
    X_list = []
    Y_list = []
    for df in list_of_fires:
      df_groups = df.groupby('occur_id')
      for idx, seq in df_groups:
          seq_len = len(seq)
          no_sequences += 1
          if seq_len > max_seq:
              max_seq = seq_len
              longest_id = seq["occur_id"].iloc[0]
          if seq_len < min_seq:
              min_seq = seq_len

          Y = seq["Y"]
          Y_list.append(Y)
          X = seq.drop(columns=["LandCover", "WUIBreach", "Month", "first_urban_point", "RemoveFlag"])#.values
          X_list.append(X)

    return no_sequences, max_seq, min_seq, longest_id, X_list, Y_list

In [ ]:
def create_sequences_from_fire_ids_twice(list_of_fires):
    import gc
    no_sequences = 0
    max_seq = 0
    min_seq = 100
    longest_id = 0
    X_list = []
    Y_list = []
    for df in list_of_fires:

      df_odd = df[df['point_id'] % 2 != 0]
      df = df[df['point_id'] % 2 != 1]

      for idx, seq in df_odd.groupby('occur_id'):
          seq_len = len(seq)
          no_sequences += 1
          if seq_len > max_seq:
              max_seq = seq_len
              longest_id = seq["occur_id"].iloc[0]
          if seq_len < min_seq:
              min_seq = seq_len

          Y = seq["Y"]
          Y_list.append(Y)
          X = seq.drop(columns=["LandCover", "WUIBreach", "Month", "first_urban_point", "RemoveFlag"])
          X_list.append(X)

      for idx, seq in df.groupby('occur_id'):
            seq_len = len(seq)
            no_sequences += 1
            if seq_len > max_seq:
                max_seq = seq_len
                longest_id = seq["occur_id"].iloc[0]
            if seq_len < min_seq:
                min_seq = seq_len

            Y = seq["Y"]
            Y_list.append(Y)
            X = seq.drop(columns=["LandCover", "WUIBreach", "Month", "first_urban_point", "RemoveFlag"])
            X_list.append(X)

    return no_sequences, max_seq, min_seq, longest_id, X_list, Y_list

In [ ]:
def create_sequences_for_training(df):
    df_groups = df.groupby('occur_id')
    no_sequences = 0
    max_seq = 0
    min_seq = 100
    longest_id = 0
    X_list = []
    Y_list = []
    for idx, seq in df_groups:
        seq_len = len(seq)
        no_sequences += 1
        if seq_len > max_seq:
            max_seq = seq_len
            longest_id = seq["occur_id"].iloc[0]
        if seq_len < min_seq:
            min_seq = seq_len

        Y = seq["Y"]
        Y_list.append(Y)
        X = seq.drop(columns=["LandCover", "WUIBreach", "Month", "first_urban_point", "RemoveFlag"])
        X_list.append(X)

    return no_sequences, max_seq, min_seq, longest_id, X_list, Y_list


In [ ]:
def train_scaler(X_list):
    import copy
    scaler = StandardScaler()
    X_list_copy = copy.deepcopy(X_list)
    for X_sequence in X_list_copy:
        X_sequence = X_sequence.drop(columns=["IsUrban", "occur_id", "point_id", "Y", "FIRE_ID"], inplace=True)
    X_concat = np.concatenate(X_list_copy)
    scaler.fit(X_concat)
    return scaler

In [ ]:
def scaler_transform(X_list, scaler):
    new_Xlist = []
    for X_sequence in X_list:
        X_sequence.drop(columns=["IsUrban", "occur_id", "point_id", "Y", "FIRE_ID"], inplace=True)
        X_values = X_sequence.values
        X_norm = scaler.transform(X_values)
        new_Xlist.append(X_norm)
    return new_Xlist

In [ ]:
all_fires_list = create_fire_id_groups(fires)
X_train_list, X_test_list = train_test_split(all_fires_list, test_size=0.3, train_size=0.7, random_state=19, shuffle=True, stratify=None)

In [ ]:
print(f"Training fires: {len(X_train_list)}")
print(f"Testing fires: {len(X_test_list)}")

Training fires: 1560
Testing fires: 669


In [ ]:
_, steps, _, _, X_train, Y_train = create_sequences_from_fire_ids_twice(X_train_list)
_, _, _, _, X_test, Y_test = create_sequences_from_fire_ids_twice(X_test_list)

In [ ]:
print(X_train[0])

                      FIRE_ID  UrbanAngle  occur_id  point_id  IsUrban  \
399743  CA3443711869619880903   62.778349      3521         1        0   
399745  CA3443711869619880903   62.778349      3521         3        0   
399747  CA3443711869619880903   62.778349      3521         5        0   
399749  CA3443711869619880903   62.778349      3521         7        0   
399751  CA3443711869619880903   62.778349      3521         9        0   
399753  CA3443711869619880903   62.778349      3521        11        0   
399755  CA3443711869619880903   62.778349      3521        13        0   
399757  CA3443711869619880903   62.778349      3521        15        0   
399759  CA3443711869619880903   62.778349      3521        17        0   
399761  CA3443711869619880903   62.778349      3521        19        0   
399763  CA3443711869619880903   62.778349      3521        21        0   
399765  CA3443711869619880903   62.778349      3521        23        0   
399767  CA3443711869619880903   62.778

In [ ]:
X_train[0].columns

Index(['FIRE_ID', 'UrbanAngle', 'occur_id', 'point_id', 'IsUrban', 'NDVI',
       'NDWI', 'NBR', 'DEM', 'aspect', 'hillshade', 'slope', 'bi', 'erc',
       'eto', 'fm100', 'fm1000', 'pr', 'rmax', 'rmin', 'th', 'pop_density',
       'tmmn', 'tmmx', 'vpd', 'vs', 'Y', 'Water', 'Very_low_rural',
       'Low_density_rural', 'Rural_cluster', 'Suburban', 'Semi_dense_urban',
       'Dense_urban', 'Urban_centre'],
      dtype='object')

In [ ]:
X_scaler = train_scaler(X_train)

                      FIRE_ID  UrbanAngle  occur_id  point_id  IsUrban  \
399743  CA3443711869619880903   62.778349      3521         1        0   
399745  CA3443711869619880903   62.778349      3521         3        0   
399747  CA3443711869619880903   62.778349      3521         5        0   
399749  CA3443711869619880903   62.778349      3521         7        0   
399751  CA3443711869619880903   62.778349      3521         9        0   
399753  CA3443711869619880903   62.778349      3521        11        0   
399755  CA3443711869619880903   62.778349      3521        13        0   
399757  CA3443711869619880903   62.778349      3521        15        0   
399759  CA3443711869619880903   62.778349      3521        17        0   
399761  CA3443711869619880903   62.778349      3521        19        0   
399763  CA3443711869619880903   62.778349      3521        21        0   
399765  CA3443711869619880903   62.778349      3521        23        0   
399767  CA3443711869619880903   62.778

In [ ]:
import os
from pickle import dump

output_dir = '/content/drive/MyDrive/'
#os.makedirs(output_dir, exist_ok=True)

dump(X_scaler, open(os.path.join(output_dir, 'wildfire_scaler_WUI.keras'), 'wb'))

In [ ]:
X_train = scaler_transform(X_train, X_scaler)

In [ ]:
print(X_train[0].shape)

(54, 30)


In [ ]:
print(X_train[0])

[[ 0.59677807 -0.45425718 -0.40726908 ... -0.06851018 -0.08069557
  -0.13497143]
 [ 0.59677807 -0.32627379 -0.30490184 ... -0.06851018 -0.08069557
  -0.13497143]
 [ 0.59677807 -0.41853115 -0.52534541 ... -0.06851018 -0.08069557
  -0.13497143]
 ...
 [ 0.59677807 -0.31022766 -0.12471519 ... -0.06851018 -0.08069557
  -0.13497143]
 [ 0.59677807 -0.24457923 -0.17757606 ... -0.06851018 -0.08069557
  -0.13497143]
 [ 0.59677807 -0.31002043 -0.11500862 ... -0.06851018 -0.08069557
  -0.13497143]]


In [ ]:
X_padded = sequence.pad_sequences(X_train, padding='post', dtype='float32', value=-9)
Y_padded = sequence.pad_sequences(Y_train, padding='post', dtype='float32', value=-9)

In [ ]:
print(len(X_padded))
print(len(Y_padded))

print(X_padded[0].shape)
print(Y_padded[0].shape)
print(X_padded.shape)
print(Y_padded.shape)

15399
15399
(101, 30)
(101,)
(15399, 101, 30)
(15399, 101)


In [ ]:
def count_Y_values(Ytrue_list):
  zero_count = 0
  one_count = 0
  null_count = 0
  all_counts = 0
  for seq in Ytrue_list:
    for i in seq:
      all_counts += 1
      if i == 0:
        zero_count += 1
      elif i == 1:
        one_count += 1
      elif i == -9:
        null_count += 1
  print(f"Zero Count: {zero_count}")
  print(f"One Count: {one_count}")
  print(f"Null Count: {null_count}")
  print(f"All Count: {all_counts}")
  return zero_count, one_count, null_count, all_counts

In [ ]:
# https://keras.io/examples/structured_data/imbalanced_classification/

counts = count_Y_values(Y_padded)
print(
    "Number of positive samples in training data: {} ({:.2f}% of total)".format(
        counts[1], 100 * float(counts[1]) / counts[3]
    )
)

Zero Count: 704770
One Count: 261898
Null Count: 588631
All Count: 1555299
Number of positive samples in training data: 261898 (16.84% of total)


In [1]:
sample_weight = (Y_padded != -9).astype("float32")
Y_padded2 = np.where(Y_padded == -9, 0, Y_padded).astype("float32")

NameError: name 'Y_padded' is not defined

In [ ]:
model = keras.models.Sequential()
model.add(keras.Input(shape=(steps, X_padded.shape[2])))
model.add(keras.layers.Masking(mask_value=-9))
model.add(LSTM(64, return_sequences=True, kernel_initializer='glorot_uniform'))
model.add(Dropout(0.4))
model.add(LSTM(32, return_sequences=True))
model.add(keras.layers.TimeDistributed(Dense(1, activation='sigmoid')))
# model.add(Dense(1)) : this would predict the final value, in this case the value at the WUI border
model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy','recall'])
model.fit(X_padded, Y_padded2, epochs=50, batch_size=1, verbose=2, sample_weight=sample_weight)

Epoch 1/50
15399/15399 - 284s - 18ms/step - accuracy: 0.8655 - loss: 0.2396 - recall: 0.6774
Epoch 2/50
15399/15399 - 271s - 18ms/step - accuracy: 0.8818 - loss: 0.1773 - recall: 0.7354
Epoch 3/50
15399/15399 - 269s - 17ms/step - accuracy: 0.8876 - loss: 0.1656 - recall: 0.7554
Epoch 4/50
15399/15399 - 338s - 22ms/step - accuracy: 0.8941 - loss: 0.1546 - recall: 0.7733
Epoch 5/50
15399/15399 - 310s - 20ms/step - accuracy: 0.8991 - loss: 0.1472 - recall: 0.7901
Epoch 6/50
15399/15399 - 263s - 17ms/step - accuracy: 0.9039 - loss: 0.1401 - recall: 0.8072
Epoch 7/50
15399/15399 - 319s - 21ms/step - accuracy: 0.9075 - loss: 0.1339 - recall: 0.8189
Epoch 8/50
15399/15399 - 259s - 17ms/step - accuracy: 0.9108 - loss: 0.1289 - recall: 0.8269
Epoch 9/50
15399/15399 - 255s - 17ms/step - accuracy: 0.9128 - loss: 0.1260 - recall: 0.8319
Epoch 10/50
15399/15399 - 259s - 17ms/step - accuracy: 0.9157 - loss: 0.1222 - recall: 0.8406
Epoch 11/50
15399/15399 - 276s - 18ms/step - accuracy: 0.9182 - loss:

In [ ]:
model.save('/content/drive/MyDrive/wildfire_model_WUI_100m.keras')



In [ ]:
import copy
X_test_ref = []
for df in X_test:
  df_ref = df.copy(deep=True)
  X_test_ref.append(df_ref)

len(X_test_ref)

6456

In [ ]:
urb_list = []
for df in X_test_ref:
  X_urb = df["IsUrban"]
  urb_list.append(X_urb)

len(urb_list)

6456

In [ ]:
X_test = scaler_transform(X_test, X_scaler)
X_test_padded = sequence.pad_sequences(X_test, padding='post', dtype='float32', value=-9)
X_test_ref_padded = sequence.pad_sequences(X_test, padding='post', dtype='float32', value=-9)
Y_test_padded = sequence.pad_sequences(Y_test, padding='post', dtype='float32', value=-9)
X_test_IsUrban_padded = sequence.pad_sequences(urb_list, padding='post', dtype='float32', value=-9)

In [ ]:
print(X_test_padded.shape)
print(Y_test_padded.shape)
print(X_test_IsUrban_padded.shape)
print(X_test_ref_padded.shape)

(6456, 101, 30)
(6456, 101)
(6456, 101)
(6456, 101, 30)


In [ ]:
# https://machinelearningmastery.com/multivariate-time-series-forecasting-lstms-keras/

# make a prediction
yhat = model.predict(X_test_padded)
threshold = 0.4



202/202 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step


In [ ]:
y_pred = np.where(yhat > threshold, 1,0).astype(int)
y = Y_test_padded.astype(int)

In [ ]:
y_pred = y_pred.reshape(y_pred.shape[0], y_pred.shape[1])

print(y_pred.shape)
print(y.shape)

(6456, 101)
(6456, 101)


In [ ]:
X_test_IsUrban_padded = X_test_IsUrban_padded.astype(int)
urb = X_test_IsUrban_padded.reshape(X_test_IsUrban_padded.shape[0], X_test_IsUrban_padded.shape[1])
print(urb.shape)

(6456, 101)


In [ ]:
print(y[0])

In [ ]:
print(y_pred[0])

In [ ]:
print(urb[3])

In [ ]:
y_list = []
y_pred_list = []
y_final = []
y_pred_final = []
y_final_lengths = []
y_list_urb = []
y_pred_list_urb = []

for y_seq, y_pred_seq, urb_seq in zip(y, y_pred, urb):
  seq_mask = y_seq != -9
  y_seq = y_seq[seq_mask]
  y_pred_seq = y_pred_seq[seq_mask]
  urb_mask_int = urb_seq[seq_mask]

  y_urb_seq = y_seq[urb_mask_int.astype(bool)]
  y_pred_urb_seq = y_pred_seq[urb_mask_int.astype(bool)]

  if len(y_urb_seq) > 0:
    y_WUI = np.max(y_urb_seq)
    y_WUI_pred = np.max(y_pred_urb_seq)
  else:
    y_WUI = 0
    y_WUI_pred = 0

  y_final.append([y_WUI])
  y_pred_final.append([y_WUI_pred])

  y_list.append(y_seq)
  y_pred_list.append(y_pred_seq)
  y_list_urb.append(y_urb_seq)
  y_pred_list_urb.append(y_pred_urb_seq)

  y_urb_values = y_seq[urb_mask_int.astype(bool)]
  y_pred_urb_values = y_pred_seq[urb_mask_int.astype(bool)]


  if np.any(urb_seq == 1):
    first_urban = np.where(urb_seq == 1)[0][0]
    y_final_lengths.append(first_urban + 1)
  else:
    y_final_lengths.append(-9)

In [ ]:
X_test_output_list = []
for test_df, y_t in zip(X_test_ref, y_pred_list):
  test_df_copy = test_df.copy()
  test_df_copy["Y_predicted"] = y_t
  X_test_output_list.append(test_df_copy)

print(X_test_output_list[0].head())

                     FIRE_ID  UrbanAngle  occur_id  point_id  IsUrban  \
75480  CA3340311726720201224 -177.095042     10584         1        0   
75482  CA3340311726720201224 -177.095042     10584         3        0   
75484  CA3340311726720201224 -177.095042     10584         5        0   
75486  CA3340311726720201224 -177.095042     10584         7        0   
75488  CA3340311726720201224 -177.095042     10584         9        0   

           NDVI      NDWI       NBR         DEM  aspect  hillshade  slope  \
75480  0.123276  0.154627 -0.002858  229.399902   282.0      223.0   14.0   
75482  0.154991  0.171198  0.017687  227.217499   258.0      217.0   12.0   
75484  0.165118  0.201804  0.008174  213.764359   203.0      195.0   11.0   
75486  0.157125  0.169961  0.033804  199.539124   301.0      221.0   15.0   
75488  0.139429  0.150008  0.055723  209.033646   269.0      209.0    9.0   

         bi   erc  eto  fm100  fm1000   pr        rmax       rmin    th  \
75480  48.0  55.0  3.5 

In [ ]:
out_df = X_test_output_list[0]
for df in X_test_output_list[1:]:
  out_df = pd.concat([out_df, df])


In [ ]:
len(out_df)

In [ ]:
out_df.to_csv('/content/drive/MyDrive/wildfire_model_df_100m.csv')

In [ ]:
print(y_final_lengths[:100])


In [ ]:
print(y_list[0])
print("\n")
print(y_pred_list[0])


In [ ]:
print(y_list_urb[1])
print("\n")
print(y_pred_list_urb[1])

In [ ]:
print(y_final)
print("\n")
print(y_pred_final)
print("\n")
print(y_final_lengths)

In [ ]:
def calculate_metrics_all_points(ytrue_list, yhat_list, min_length, max_length, Breach=False):
  TP_count = 0
  FP_count = 0
  TN_count = 0
  FN_count = 0
  for y_seq, y_pred_seq in zip(ytrue_list, yhat_list):
    if len(y_seq) > min_length and len(y_seq) <= max_length:
      for i in range(len(y_seq)):
        if y_seq[i] == 1 and y_pred_seq[i] == 1:
          TP_count += 1
        elif y_seq[i] == 0 and y_pred_seq[i] == 1:
          FP_count += 1
        elif y_seq[i] == 1 and y_pred_seq[i] == 0:
          FN_count += 1
        elif y_seq[i] == 0 and y_pred_seq[i] == 0:
          TN_count += 1

    total_rows = TP_count + FP_count + TN_count + FN_count
    try:
      accuracy = round((TP_count + TN_count) / total_rows, 2)
    except:
      accuracy = 0

    # precision tp / (tp + fp)
    try:
      precision = round(TP_count / (TP_count + FP_count),2)
    except:
      precision = 0

    # recall: tp / (tp + fn)
    try:
      recall = round(TP_count / (TP_count + FN_count), 2)
    except:
      recall = 0

    # f1: 2 tp / (2 tp + fp + fn)
    try:
      f1 = round(2 * ((precision * recall)/ (precision + recall)), 2)
    except:
      f1 = 0


    confusion_matrix = [TP_count, FP_count, FN_count, TN_count]


  return accuracy, precision, recall, f1, confusion_matrix


In [ ]:
def test_calculate_metrics_all_points(ytrue_list, yhat_list, min_distance, max_distance, Breach=False):
  TP_count = 0
  FP_count = 0
  TN_count = 0
  FN_count = 0
  for y_seq, y_pred_seq in zip(ytrue_list, yhat_list):
    for i in range(len(y_seq)):
      if i >= min_distance and i < max_distance:
        if y_seq[i] == 1 and y_pred_seq[i] == 1:
          TP_count += 1
        elif y_seq[i] == 0 and y_pred_seq[i] == 1:
          FP_count += 1
        elif y_seq[i] == 1 and y_pred_seq[i] == 0:
          FN_count += 1
        elif y_seq[i] == 0 and y_pred_seq[i] == 0:
          TN_count += 1

    total_rows = TP_count + FP_count + TN_count + FN_count
    try:
      accuracy = round((TP_count + TN_count) / total_rows, 2)
    except:
      accuracy = 0

    # precision tp / (tp + fp)
    try:
      precision = round(TP_count / (TP_count + FP_count),2)
    except:
      precision = 0

    # recall: tp / (tp + fn)
    try:
      recall = round(TP_count / (TP_count + FN_count), 2)
    except:
      recall = 0

    # f1: 2 tp / (2 tp + fp + fn)
    try:
      f1 = round(2 * ((precision * recall)/ (precision + recall)), 2)
    except:
      f1 = 0


    confusion_matrix = [TP_count, FP_count, FN_count, TN_count]


  return accuracy, precision, recall, f1, confusion_matrix

In [ ]:
model_accuracy, model_precision, model_recall, model_f1, model_confusion = calculate_metrics_all_points(y_list, y_pred_list, 0, X_test_padded.shape[0])
print(f"Accuracy: {model_accuracy}")
print(f"Recall: {model_recall}")
print(f"Precision: {model_precision}")
print(f"F1 score: {model_f1}")
print(f"Confusion matrix: {model_confusion}")

Accuracy: 0.83
Recall: 0.79
Precision: 0.67
F1 score: 0.73
Confusion matrix: [88746, 43933, 23074, 246912]


In [ ]:
urban_fire_accuracy, urban_fire_precision, urban_fire_recall, urban_fire_f1, urban_fire_confusion = calculate_metrics_all_points(y_list_urb, y_pred_list_urb, 0, X_test_padded.shape[0])

# this is a fire / non fire prediction, not a WUI breach prediction
# for urban points, did the fire spread there?

print(f"Accuracy (urban fire): {urban_fire_accuracy}")
print(f"Recall (urban fire): {urban_fire_recall}")
print(f"Precision (urban fire): {urban_fire_precision}")
print(f"F1 score (urban fire): {urban_fire_f1}")
print(f"Confusion matrix (urban fire): {urban_fire_confusion}")

Accuracy (urban fire): 0.94
Recall (urban fire): 0.64
Precision (urban fire): 0.38
F1 score (urban fire): 0.48
Confusion matrix (urban fire): [1452, 2359, 832, 45319]


In [ ]:
WUI_breach_accuracy, WUI_breach_precision, WUI_breach_recall, WUI_breach_f1, WUI_breach_confusion = calculate_metrics_all_points(y_final, y_pred_final, 0, X_test_padded.shape[0])

print(f"WUI Breach Accuracy: {WUI_breach_accuracy}")
print(f"WUI Breach Recall: {WUI_breach_recall}")
print(f"WUI Breach Precision: {WUI_breach_precision}")
print(f"WUI Breach F1 score: {WUI_breach_f1}")
print(f"WUI Breach Confusion matrix: {WUI_breach_confusion}")

WUI Breach Accuracy: 0.9
WUI Breach Recall: 0.73
WUI Breach Precision: 0.47
WUI Breach F1 score: 0.57
WUI Breach Confusion matrix: [429, 479, 156, 5392]


In [ ]:
def display_confusion_matrix(confusion_matrix, min_distance, max_distance):
  header = f"{min_distance}km to {max_distance}km"
  c_f = pd.DataFrame(columns=[header, 'True', 'False'])

  new_row_positive = pd.DataFrame([["Positive", confusion_matrix[0], confusion_matrix[1]]], columns=c_f.columns)
  c_f = pd.concat([c_f, new_row_positive], ignore_index =True)

  new_row_negative = pd.DataFrame([["Negative", confusion_matrix[3], confusion_matrix[2]]], columns=c_f.columns)
  c_f = pd.concat([c_f, new_row_negative], ignore_index =True)

  return c_f


In [ ]:
def calculate_metrics_by_distance(ytrue_list, yhat_list, interval):
  min_ = 0
  start_ = 0 + interval
  max_ = start_
  all_cm = []
  all_point_stats = pd.DataFrame(columns=['Min_Kilometres', 'Max_Kilometres', 'Accuracy', 'Precision', 'Recall', 'F1'])
  for max_ in range(start_, 105, interval):
    accuracy, precision, recall, f1, conf = test_calculate_metrics_all_points(ytrue_list, yhat_list, min_, max_)
    new_row = pd.DataFrame([[(min_/10), (max_/10), accuracy, precision, recall, f1]], columns=all_point_stats.columns)
    all_point_stats = pd.concat([all_point_stats, new_row], ignore_index =True)
    conf_matrix = display_confusion_matrix(conf, min_/10, max_/10)
    all_cm.append(conf_matrix)
    min_ += interval
    max_ += interval

  return all_point_stats, all_cm


In [ ]:
def calculate_WUI_breach_metrics_by_distance(ytrue_list, yhat_list, y_length_list, interval):
  min_ = 0
  start_ = 0 + interval
  max_ = start_
  all_cm = []
  all_point_stats = pd.DataFrame(columns=['Min_Kilometres', 'Max_Kilometres', 'Accuracy', 'Precision', 'Recall', 'F1'])

  for max_ in range(start_, 105, interval):
    y_check = []
    y_pred_check = []
    for y, y_pred, y_length in zip(ytrue_list, yhat_list, y_length_list):
      if y_length > min_ and y_length <= max_:
        y_check.append(y)
        y_pred_check.append(y_pred)
    accuracy, precision, recall, f1, conf = test_calculate_metrics_all_points(y_check, y_pred_check, 0, X_test_padded.shape[0])
    new_row = pd.DataFrame([[(min_/10), (max_/10), accuracy, precision, recall, f1]], columns=all_point_stats.columns)
    all_point_stats = pd.concat([all_point_stats, new_row], ignore_index =True)
    conf_matrix = display_confusion_matrix(conf, min_/10, max_/10)
    all_cm.append(conf_matrix)
    min_ += interval
    max_ += interval

  return all_point_stats, all_cm


In [ ]:
distance_stats, distance_cm = calculate_metrics_by_distance(y_list, y_pred_list, 10)

print(distance_stats)
print("\n")
for c in distance_cm:
  print(c)
  print("\n")

/tmp/ipykernel_6971/1875484337.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_point_stats = pd.concat([all_point_stats, new_row], ignore_index =True)


   Min_Kilometres  Max_Kilometres  Accuracy  Precision  Recall    F1
0             0.0             1.0      0.89       0.91    0.97  0.94
1             1.0             2.0      0.63       0.60    0.75  0.67
2             2.0             3.0      0.70       0.34    0.49  0.40
3             3.0             4.0      0.82       0.22    0.32  0.26
4             4.0             5.0      0.91       0.23    0.30  0.26
5             5.0             6.0      0.94       0.27    0.29  0.28
6             6.0             7.0      0.96       0.31    0.29  0.30
7             7.0             8.0      0.98       0.38    0.34  0.36
8             8.0             9.0      0.99       0.47    0.49  0.48
9             9.0            10.0      1.00       1.00    1.00  1.00


  0.0km to 1.0km   True False
0       Positive  56182  5774
1       Negative    670  1570


  1.0km to 2.0km   True  False
0       Positive  23099  15360
1       Negative  15623   7524


  2.0km to 3.0km   True  False
0       Positive   59

In [ ]:
breach_distance_stats, breach_distance_cm = calculate_WUI_breach_metrics_by_distance(y_final, y_pred_final, y_final_lengths, 25)

print(breach_distance_stats)
print("\n")
for c in breach_distance_cm:
  print(c)
  print("\n")

   Min_Kilometres  Max_Kilometres  Accuracy  Precision  Recall    F1
0             0.0             2.5      0.72       0.62    0.81  0.70
1             2.5             5.0      0.87       0.12    0.36  0.18
2             5.0             7.5      0.97       0.19    0.42  0.26
3             7.5            10.0      0.99       0.40    0.60  0.48


  0.0km to 2.5km True False
0       Positive  388   239
1       Negative  445    92


  2.5km to 5.0km  True False
0       Positive    27   197
1       Negative  1583    49


  5.0km to 7.5km  True False
0       Positive     8    34
1       Negative  1573    11


  7.5km to 10.0km  True False
0        Positive     6     9
1        Negative  1652     4




/tmp/ipykernel_6971/2785806771.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_point_stats = pd.concat([all_point_stats, new_row], ignore_index =True)


In [ ]:
all_confusion_matrix = display_confusion_matrix(model_confusion, 0, 100)
print(all_confusion_matrix)

  0km to 100km    True  False
0     Positive   88746  43933
1     Negative  246912  23074
